# Download real videos and upload to HF

Sometimes modelscope block access for the Ukrainian IP, so Google Colab helps with this issue

In [1]:
!pip install -q huggingface_hub requests tqdm
!apt-get install -qq unrar

In [2]:
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [3]:
import os, shutil, tarfile, zipfile, random, subprocess
from pathlib import Path
import requests
from tqdm import tqdm
from huggingface_hub import upload_folder

In [4]:
REPO_ID = 'ironiss/PhysVidDetect-v1'
MODELSCOPE_PATH = 'https://www.modelscope.cn/api/v1/datasets/cccnju/GenVideo-100K/repo?Source=SDK&Revision=master&FilePath={}&View=False'
K400_TRAIN_PATH_URL = 'https://s3.amazonaws.com/kinetics/400/train/k400_train_path.txt'
MSRVTT_URL = 'https://www.robots.ox.ac.uk/~maxbain/frozen-in-time/data/MSRVTT.zip'
UCF101_URL = 'https://www.crcv.ucf.edu/datasets/human-actions/ucf101/UCF101.rar'
GENVIDEO_PARTS = ['Real_part_aa', 'Real_part_ab', 'Real_part_ac']
VIDEO_EXTS = {'.mp4', '.mov', '.webm', '.mkv', '.avi'}
SEED = 42

WORK_DIR = Path('/content/real_download')
WORK_DIR.mkdir(parents=True, exist_ok=True)

In [6]:
import urllib3, warnings
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
warnings.filterwarnings('ignore')

In [8]:
print("MSRVTT:")
r = requests.head(MSRVTT_URL, timeout=30, allow_redirects=True)
print(f"  {r.status_code}  {int(r.headers.get('Content-Length',0))/1e9:.2f} GB")

print("\nGenVideo-Real parts:")
for p in GENVIDEO_PARTS:
    r = requests.head(MODELSCOPE_PATH.format(p), timeout=30, verify=False, allow_redirects=True)
    print(f"  {p:15} {r.status_code}  {int(r.headers.get('Content-Length',0))/1e9:.2f} GB")

print("\nKinetics-400 URL list:")
r = requests.get(K400_TRAIN_PATH_URL, timeout=30, verify=False)
urls = [l.strip() for l in r.text.splitlines() if l.strip()]
print(f"  {r.status_code}  {len(urls)} parts, first: {urls[0] if urls else 'NONE'}")

print("\nUCF101:")
# r = requests.head(UCF101_URL, timeout=30, allow_redirects=True)
r = requests.head(UCF101_URL, timeout=30, allow_redirects=True, verify=False)
print(f"  {r.status_code}  {int(r.headers.get('Content-Length',0))/1e9:.2f} GB")


MSRVTT:
  200  6.55 GB

GenVideo-Real parts:
  Real_part_aa    200  32.21 GB
  Real_part_ab    200  32.21 GB
  Real_part_ac    200  29.12 GB

Kinetics-400 URL list:
  200  242 parts, first: https://s3.amazonaws.com/kinetics/400/train/part_0.tar.gz

UCF101:
  200  6.93 GB


## Config

In [5]:
SOURCES_TO_DOWNLOAD = ['GenVideo-Real', 'Kinetics-400', 'UCF101']
GENVIDEO_REAL_TARGET = 3000
OTHER_TARGETS = 7000

In [ ]:
def stream_download(url, dest, verify=True):
    with requests.get(url, stream=True, timeout=300, verify=verify) as r:
        r.raise_for_status()
        total = int(r.headers.get('Content-Length', 0))
        with open(dest, 'wb') as f, tqdm(total=total, unit='B', unit_scale=True, desc=dest.name) as bar:
            for chunk in r.iter_content(chunk_size=1024*1024):
                if chunk:
                    f.write(chunk)
                    bar.update(len(chunk))


def find_videos(root):
    out = []
    for r, _, files in os.walk(root):
        for f in files:
            if Path(f).suffix.lower() in VIDEO_EXTS:
                out.append(Path(r) / f)
    return out


def sample_videos(videos, n):
    if len(videos) <= n:
        return videos
    random.seed(SEED)
    return sorted(random.sample(videos, n))


def upload_and_cleanup(out_dir, name):

    print(f'Upload videos to real/{name}/ on HF')
    
    upload_folder(
        folder_path=str(out_dir),
        path_in_repo=f'real/{name}',
        repo_id=REPO_ID,
        repo_type='dataset',
    )
    shutil.rmtree(out_dir, ignore_errors=True)


def copy_flat(videos, out_dir):
    out_dir.mkdir(parents=True, exist_ok=True)
    for v in tqdm(videos, desc='copying'):
        dst = out_dir / v.name
        if dst.exists():
            dst = out_dir / f'{v.parent.name}_{v.name}'
        shutil.copy2(v, dst)

In [ ]:
def process_msrvtt():
    archive = WORK_DIR / 'MSRVTT.zip'
    extract_dir = WORK_DIR / 'extracted_msrvtt'
    out_dir = WORK_DIR / 'upload_msrvtt'

    stream_download(MSRVTT_URL, archive, verify=True)
    with zipfile.ZipFile(archive, 'r') as zf:
        zf.extractall(extract_dir)
    archive.unlink(missing_ok=True)

    videos = find_videos(extract_dir)
    videos = sample_videos(videos, OTHER_TARGETS)
    copy_flat(videos, out_dir)
    shutil.rmtree(extract_dir, ignore_errors=True)
    upload_and_cleanup(out_dir, 'MSRVTT')


def process_genvideo_real():
    parts_dir = WORK_DIR / 'genvideo_parts'
    parts_dir.mkdir(parents=True, exist_ok=True)
    extract_dir = WORK_DIR / 'extracted_genvideo'
    extract_dir.mkdir(parents=True, exist_ok=True)
    out_dir = WORK_DIR / 'upload_genvideo'

    for part in GENVIDEO_PARTS:
        url = MODELSCOPE_PATH.format(part)
        local = parts_dir / part
        try:
            stream_download(url, local, verify=False)
        except Exception as e:
            print(f'  {part} failed: {e}')

    part_paths = [str(parts_dir / p) for p in GENVIDEO_PARTS if (parts_dir / p).exists()]
    cmd = f"cat {' '.join(part_paths)} | tar xzf - -C {extract_dir}"
    proc = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if proc.returncode != 0:
        print(f'exit with {proc.returncode}')
    shutil.rmtree(parts_dir, ignore_errors=True)

    videos = find_videos(extract_dir)
    videos = sample_videos(videos, GENVIDEO_REAL_TARGET)
    copy_flat(videos, out_dir)
    shutil.rmtree(extract_dir, ignore_errors=True)
    upload_and_cleanup(out_dir, 'GenVideo-Real')


def process_k400():
    parts_dir = WORK_DIR / 'k400_parts'
    parts_dir.mkdir(parents=True, exist_ok=True)
    extract_dir = WORK_DIR / 'extracted_k400'
    extract_dir.mkdir(parents=True, exist_ok=True)
    out_dir = WORK_DIR / 'upload_k400'

    resp = requests.get(K400_TRAIN_PATH_URL, timeout=40, verify=False)
    resp.raise_for_status()
    urls = [l.strip() for l in resp.text.splitlines() if l.strip()]

    videos = []
    for url in urls:
        if len(videos) >= OTHER_TARGETS:
            break
        local = parts_dir / Path(url).name
        try:
            stream_download(url, local, verify=True)
        except Exception as e:
            print(f'  {local.name} failed: {e}')
            local.unlink(missing_ok=True)
            continue
        try:
            with tarfile.open(local, 'r:gz') as tf:
                for m in tf.getmembers():
                    if m.isfile() and Path(m.name).suffix.lower() in VIDEO_EXTS:
                        tf.extract(m, path=extract_dir)
                        videos.append(extract_dir / m.name)
                        if len(videos) >= OTHER_TARGETS:
                            break
        finally:
            local.unlink(missing_ok=True)

    shutil.rmtree(parts_dir, ignore_errors=True)
    videos = sample_videos(videos, OTHER_TARGETS)
    copy_flat(videos, out_dir)
    shutil.rmtree(extract_dir, ignore_errors=True)
    upload_and_cleanup(out_dir, 'Kinetics-400')


def process_ucf101():
    archive = WORK_DIR / 'UCF101.rar'
    extract_dir = WORK_DIR / 'extracted_ucf101'
    extract_dir.mkdir(parents=True, exist_ok=True)
    out_dir = WORK_DIR / 'upload_ucf101'

    stream_download(UCF101_URL, archive, verify=True)

    print('extracting RAR...')
    r = subprocess.run(
        ['unrar', 'x', '-o-', str(archive), str(extract_dir) + '/'],
        capture_output=True, text=True,
    )
    if r.returncode != 0:
        print(r.stderr[:500])
        raise RuntimeError('unrar failed')
    archive.unlink(missing_ok=True)

    videos = find_videos(extract_dir)
    videos = sample_videos(videos, OTHER_TARGETS)
    copy_flat(videos, out_dir)
    shutil.rmtree(extract_dir, ignore_errors=True)
    upload_and_cleanup(out_dir, 'UCF101')

In [ ]:
HANDLERS = {
    'MSRVTT': process_msrvtt,
    'GenVideo-Real': process_genvideo_real,
    'Kinetics-400': process_k400,
    'UCF101': process_ucf101,
}

In [ ]:
for src in SOURCES_TO_DOWNLOAD:
    print(f'Processing source: {src}')
    if src not in HANDLERS:
        continue
    HANDLERS[src]()

shutil.rmtree(WORK_DIR, ignore_errors=True)